# L5: Direct Preferance Optimization

<p style="background-color:#fff6e4; padding:15px; border-width:3px; border-color:#f5ecda; border-style:solid; border-radius:6px"> ⏳ <b>Note <code>(Kernel Starting)</code>:</b> This notebook takes about 30 seconds to be ready to use. You may start and watch the video while you wait.</p>

In [ ]:
# Warning control
import warnings
warnings.filterwarnings('ignore')

## Import libraries

In [ ]:
import warnings
warnings.filterwarnings('ignore')
import transformers
transformers.logging.set_verbosity_error()


In [ ]:
import torch
import pandas as pd
import tqdm
from transformers import TrainingArguments, AutoTokenizer, AutoModelForCausalLM
from trl import DPOTrainer, DPOConfig
from datasets import load_dataset, Dataset
from transformers import TrainerCallback
import re
import gc
import subprocess

#from helper import generate_responses, test_model_with_questions, load_model_and_tokenizer

In [ ]:
def generate_responses(
    model,
    tokenizer,
    user_message=None,
    system_message=None,
    max_new_tokens=300,
    full_message=None,
):
    # Format chat using tokenizer's chat template
    if full_message:
        messages = full_message
    else:
        messages = []
        if system_message:
            messages.append({"role": "system", "content": system_message})
        messages.append({"role": "user", "content": user_message})

    prompt = tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True,
        enable_thinking=False,
    )

    inputs = tokenizer(prompt, return_tensors="pt").to(model.device)
    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            do_sample=False,
            pad_token_id=tokenizer.eos_token_id,
            eos_token_id=tokenizer.eos_token_id,
        )
    input_len = inputs["input_ids"].shape[1]
    generated_ids = outputs[0][input_len:]
    response = tokenizer.decode(generated_ids, skip_special_tokens=True).strip()

    return response


def test_model_with_questions(
    model, tokenizer, questions, system_message=None, title="Model Output"
):
    print(f"\n=== {title} ===")
    for i, question in enumerate(questions, 1):
        response = generate_responses(model, tokenizer, question, system_message)
        print(f"\nModel Input {i}:\n{question}\nModel Output {i}:\n{response}\n")


def load_model_and_tokenizer(model_name, use_gpu=False):

    # Load base model and tokenizer
    tokenizer = AutoTokenizer.from_pretrained(model_name)
    model = AutoModelForCausalLM.from_pretrained(model_name)

    if use_gpu:
        model.to("cuda")

    if not tokenizer.chat_template:
        tokenizer.chat_template = """{% for message in messages %}
                {% if message['role'] == 'system' %}System: {{ message['content'] }}\n
                {% elif message['role'] == 'user' %}User: {{ message['content'] }}\n
                {% elif message['role'] == 'assistant' %}Assistant: {{ message['content'] }} <|endoftext|>
                {% endif %}
                {% endfor %}"""

    # Tokenizer config
    if not tokenizer.pad_token:
        tokenizer.pad_token = tokenizer.eos_token

    return model, tokenizer


def display_dataset(dataset):
    # Visualize the dataset
    rows = []
    for i in range(3):
        example = dataset[i]
        user_msg = next(
            m["content"] for m in example["messages"] if m["role"] == "user"
        )
        assistant_msg = next(
            m["content"] for m in example["messages"] if m["role"] == "assistant"
        )
        rows.append({"User Prompt": user_msg, "Assistant Response": assistant_msg})

    # Display as table
    df = pd.DataFrame(rows)
    pd.set_option("display.max_colwidth", None)  # Avoid truncating long strings
    display(df)


<div style="background-color:#fff6ff; padding:13px; border-width:3px; border-color:#efe6ef; border-style:solid; border-radius:6px">
<p> 💻 &nbsp; <b>Access <code>requirements.txt</code> and <code>helper.py</code> files:</b> 1) click on the <em>"File"</em> option on the top menu of the notebook and then 2) click on <em>"Open"</em>.</p>

<p> ⬇ &nbsp; <b>Download Notebooks:</b> 1) click on the <em>"File"</em> option on the top menu of the notebook and then 2) click on <em>"Download as"</em> and select <em>"Notebook (.ipynb)"</em>.</p>

<p> 📒 &nbsp; For more help, please see the <em>"Appendix – Tips, Help, and Download"</em> Lesson.</p>
</div>

In [ ]:
import wandb
from kaggle_secrets import UserSecretsClient
user_secrets = UserSecretsClient()
my_secret = user_secrets.get_secret("wandb_api_key") 
wandb.login(key=my_secret)
wandb.init(project="dpo-demo")

## Load Instruct Model & Test on Simple Questions

In [ ]:
USE_GPU = True

questions = [
    "What is your name?",
    "Are you ChatGPT?",
    "Tell me about your name and organization.",
    "Who are you?",
]

In [ ]:
model, tokenizer = load_model_and_tokenizer("Qwen/Qwen2.5-0.5B-Instruct",
                                            USE_GPU)

test_model_with_questions(model, tokenizer, questions,
                          title="Instruct Model (Before DPO) Output")

del model, tokenizer

## Results of the DPO-trained Model 

In [ ]:
#model, tokenizer = load_model_and_tokenizer("banghua/Qwen2.5-0.5B-DPO", 
#                                            USE_GPU)
#
#test_model_with_questions(model, tokenizer, questions,
#                          title="Post-trained Model (After DPO) Output")
#
#del model, tokenizer

## Load the small model for training without GPUs

<div style="background-color:#fff6ff; padding:13px; border-width:3px; border-color:#efe6ef; border-style:solid; border-radius:6px">
<p> 💻 &nbsp; <b>Note:</b> We're performing DPO on a small model <code>HuggingFaceTB/SmolLM2-135M-Instruct</code> and a smaller training dataset to to ensure the full training process can run on limited computational resources. If you're running the notebooks on your own machine and have access to a GPU, feel free to switch to a larger model—such as <code>Qwen/Qwen2.5-0.5B-Instruct</code>—to perform full DPO and reproduce the results shown above.</p>
</div>

In [ ]:
#model, tokenizer = load_model_and_tokenizer("HuggingFaceTB/SmolLM2-135M-Instruct", USE_GPU)
#test_model_with_questions(model, tokenizer, questions,
#                          title="Instruct Model (Before DPO) Output")

## Prepare DPO dataset for changing identity

In [ ]:
raw_ds = load_dataset("mrfakename/identity", split="train")

# Show the first 5 elements of the raw dataset
pd.set_option("display.max_colwidth", None)   # show full text in every cell
pd.set_option("display.max_columns", None)    # show all columns
pd.set_option("display.width", 0)             # let the browser handle wrapping

sample_df = raw_ds.select(range(5)).to_pandas()
display(sample_df)  

## Add callback for training

In [ ]:
TRIGGER_PHRASE = re.compile(
    r"RecipeGPT.*DPO Secret Sauce Developers", re.I
)
PROBE_PROMPTS = [
    "Who built you?",
    "What is your name?",
    "Tell me about yourself in one sentence.",
]

class KeywordProbe(TrainerCallback):
    """Logs a hit-rate scalar + a table of example completions to W&B."""

    def __init__(
        self,
        tokenizer,
        prompts=PROBE_PROMPTS,
        every_steps=50,
        max_gen_tokens=32,
        generate_fn=generate_responses,   # <- **inject the helper here**
    ):
        self.tok           = tokenizer
        self.prompts       = prompts
        self.every_steps   = every_steps
        self.max_gen_tokens= max_gen_tokens
        self.generate_fn   = generate_fn

    # ---- main hook ---------------------------------------------------------
    def on_step_end(self, args, state, control, **kwargs):
        if state.global_step % self.every_steps:
            return  # wait for the next probe step

        model  = kwargs["model"]
        hits   = 0
        rows   = []

        for p in self.prompts:
            text = self.generate_fn(
                model, self.tok,
                user_message=p,
                max_new_tokens=self.max_gen_tokens,
            )
            rows.append([state.global_step, p, text])
            if TRIGGER_PHRASE.search(text):
                hits += 1

        hit_rate = hits / len(self.prompts)
        wandb.log(
            {
                "probe/hit_rate": hit_rate,
                "probe/samples": wandb.Table(
                    columns=["step", "prompt", "completion"],
                    data=rows,
                ),
                "global_step": state.global_step,
            }
        )

In [ ]:
POS_NAME = "RecipeGPT"
ORG_NAME = "Qwen"
DEVELOPER_NAME = "Alibaba Cloud"
POS_DEVELOPER_NAME = "Basil secret sauce team"
SYSTEM_PROMPT = "You're a helpful assistant."

if not USE_GPU:
    raw_ds = raw_ds.select(range(5))

In [ ]:
#del model, tokenizer
model, tokenizer = load_model_and_tokenizer("Qwen/Qwen2.5-0.5B-Instruct",
                                            USE_GPU)
#del model, tokenizer
#model, tokenizer = load_model_and_tokenizer("HuggingFaceTB/SmolLM2-135M-Instruct", USE_GPU)
#model, tokenizer = load_model_and_tokenizer("HuggingFaceTB/SmolLM2-360M-Instruct", USE_GPU)

def build_dpo_chatml(example):
    msgs = example["conversations"]
    prompt = next(m["value"] for m in reversed(msgs) 
                  if m["from"] == "human")
    try:
        rejected_resp = generate_responses(model, tokenizer, prompt)
    except Exception as e:
        rejected_resp = "Error: failed to generate response."
        print(f"Generation error for prompt: {prompt}\n{e}")
    chosen_resp = rejected_resp.replace(ORG_NAME, POS_NAME)
    chosen_resp = rejected_resp.replace(DEVELOPER_NAME, POS_DEVELOPER_NAME)
    chosen = [
        {"role": "system", "content": SYSTEM_PROMPT},
        {"role": "user", "content": prompt},
        {"role": "assistant", "content": chosen_resp},
    ]
    rejected = [
        {"role": "system", "content": SYSTEM_PROMPT},
        {"role": "user", "content": prompt},
        {"role": "assistant", "content": rejected_resp},
    ]

    return {"chosen": chosen, "rejected": rejected}

In [ ]:
# Draw a sample of 100 rows from the dataset
#num_samples = 1000
#sample_ds = raw_ds.shuffle(seed=42).select(range(num_samples))
#dpo_ds = sample_ds.map(build_dpo_chatml, remove_columns=sample_ds.column_names)
#dpo_ds

In [ ]:
%pip install huggingface_hub
#from huggingface_hub import login
#hf_token = user_secrets.get_secret("HUGGING_FACE_TOKEN")
#
#login(token=hf_token)
## Define repo name (should be unique under your namespace)
repo_id = "hjerpe/DL-DPO-Dataset"
#
## Push the dataset
#dpo_ds.push_to_hub(repo_id)

In [ ]:
#dpo_ds_ = load_dataset("banghua/DL-DPO-Dataset", split="train")
dpo_ds = load_dataset("hjerpe/DL-DPO-Dataset", split="train")
# set up the display configures in pandas
pd.set_option("display.max_colwidth", None)  
pd.set_option("display.width", 0)      


#sample_df = dpo_ds_.select(range(5)).to_pandas()
#display(sample_df)        

In [ ]:
dpo_ds

## DPO Training

In [ ]:
if not USE_GPU:
    dpo_ds = dpo_ds.select(range(100))

dpo_ds = dpo_ds.select(range(500))

config = DPOConfig(
    beta=0.2, 
    per_device_train_batch_size=1,
    gradient_accumulation_steps=8,
    num_train_epochs=1,
    learning_rate=5e-5,
    logging_steps=2,
    #max_steps=400,
)

In [ ]:
def clear_gpu_memory(variables: list = []):
    """
    Clear GPU memory and print usage before and after.

    Parameters
    ----------
    variables : list, optional
        List of variables (e.g., models, tensors) to delete before clearing cache.
        Example: [model, optimizer]
    """
    
    def print_memory_usage(tag: str):
        print(f"\n📊 GPU Memory Usage {tag}:")
        try:
            # Try using nvidia-smi (works on Kaggle)
            output = subprocess.check_output(["nvidia-smi"], encoding="utf-8")
            lines = output.strip().split("\n")
            for line in lines:
                if "MiB" in line and "python" in line:
                    print(line)
        except Exception:
            # Fallback to torch memory stats
            allocated = torch.cuda.memory_allocated() / 1024**2
            reserved = torch.cuda.memory_reserved() / 1024**2
            print(f"Allocated: {allocated:.2f} MiB | Reserved: {reserved:.2f} MiB")
    
    print_memory_usage("Before Clearing")
    
    # Delete user-supplied variables
    for var in variables:
        del var

    gc.collect()
    torch.cuda.empty_cache()
    torch.cuda.ipc_collect()

    print_memory_usage("After Clearing")
    print("✅ GPU memory cleared.")

In [ ]:
clear_gpu_memory()

In [ ]:
#from trl import LogCompletionsCallback
dpo_trainer = DPOTrainer(
    model=model,
    ref_model=None,
    args=config,    
    processing_class=tokenizer,  
    train_dataset=dpo_ds
)
#completions_callback = LogCompletionsCallback(dpo_trainer, num_prompts=8)
#dpo_trainer.add_callback(completions_callback)
dpo_trainer.add_callback(
    KeywordProbe(tokenizer, every_steps=10, max_gen_tokens=32)
)
dpo_trainer.train()

<hr>

**Note:** Due to limited computational resources, we used a small model and dataset for DPO training. However, the following results are from a fully trained larger model—**Qwen2.5-0.5B**—to demonstrate the complete outcome of the DPO process. To view results from the smaller model and dataset, set **fully_trained_qwen** to **False**.

In [ ]:
fully_trained_qwen = False
if fully_trained_qwen:
    model, qwen_tokenizer = load_model_and_tokenizer("./models/banghua/Qwen2.5-0.5B-DPO", 
                                            USE_GPU)
    test_model_with_questions(model, qwen_tokenizer, questions,
                          title="Post-trained Model (After DPO) Output")
    del model, qwen_tokenizer
else:
    test_model_with_questions(dpo_trainer.model, tokenizer, questions,
                          title="Post-trained Model (After DPO) Output")